## Теория: Изменяемая конфигурация
---

Речь идёт о паттерне, когда у объекта есть настройки по умолчанию (заданные при создании), но при каждом вызове метода можно переопределить отдельные параметры.

#### **Пример:**
```python
request = Request(timeout=1)
response = request.get("http://example.com")

# Передаем дополнительный параметр при вызове метода
response = request.get("http://example.com", timeout=10)
response = request.get("http://example.com")  # таймаут остается равным 1
```
##### **Что здесь происходит**

- **`request = Request(timeout=1)`** — при создании экземпляра `Request` мы задаём базовый таймаут 1 секунда. Это значение сохраняется внутри объекта (например, `в self.timeout`) и будет использоваться по умолчанию.

- **`request.get("http://example.com")`** — вызываем метод без явного указания таймаута. Метод берёт значение из объекта: `self.timeout` → 1.

- **`request.get("http://example.com", timeout=10)`** — передаём параметр явно. Метод использует это значение `(10)` только для этого запроса.

- **`request.get("http://example.com")`** — снова без параметра. Метод опять берёт значение из объекта → 1. То есть переопределение не «залипает»: оно действует только на один вызов.


#### **Типичная реализация метода `get` выглядит примерно так:**
```python
class Request:
    def __init__(self, timeout=None):
        # Сохраняем базовый таймаут (None означает «нет значения по умолчанию»)
        self.timeout = timeout

    def get(self, url, timeout=None):
        # Если при вызове передали timeout, используем его.
        # Иначе берём из self.timeout (настройки объекта).
        effective_timeout = timeout if timeout is not None else self.timeout
        
        # Дальше используем effective_timeout для выполнения запроса
        return self._execute(url, timeout=effective_timeout)

    def _execute(self, url, timeout):
        # Здесь реальная логика отправки запроса с данным таймаутом
        ...
```

#### **Важные детали**

- **Приоритет:** аргумент, переданный в метод `(timeout=10)`, имеет приоритет над значением из объекта `(self.timeout)`.

- **Изоляция:** изменение параметра при вызове не меняет состояние объекта. Поэтому следующий вызов без параметра снова использует старое значение.

- **`None`** как маркер: часто используют `None` для обозначения «не передано». Это позволяет отличать «я не указал значение» от «я явно указал значение, даже если оно такое же».



Частые ошибки и нюансы

- **Изменяемые значения по умолчанию:** никогда не используй изменяемые объекты (списки, словари) как значения по умолчанию в сигнатуре метода. Это классическая ошибка в Python.

- **Перекрытие имён:** если у тебя в классе есть атрибут `timeout` и параметр метода тоже `timeout`, следи, чтобы не запутаться, какое значение ты используешь.

- **Документация:** для сложных объектов (как твой PasswordValidator ранее) полезно явно описывать, какие параметры можно переопределять при вызове, а какие — только при создании.

---

##### **Уровни конфигурации и их приоритет**

- Значения по умолчанию в классе/библиотеке (часто неявно): например, библиотека сама по себе считает, что таймаут по умолчанию — 30 секунд.

- Конфигурация экземпляра `(instance config)`: `Request(timeout=1)` — ты задаёшь настройки, которые живут в объекте и применяются ко всем обычным вызовам.

- Конфигурация вызова `(call-time config)`: `request.get(..., timeout=10)` — переопределение только для этого конкретного вызова.

Приоритет всегда: `call-time > instance config > defaults`.

---